In [1]:
# Standard training (next-token prediction only):
# Position i → predict token i+1          ← ONE supervision signal

# MTP training (multi-token prediction):
# Position i → predict token i+1          ← main loss (L_main)
# Position i → predict token i+2          ← MTP module 1 (L_MTP_1)
# Position i → predict token i+3          ← MTP module 2 (L_MTP_2)

- Why does this help?

- Three reasons — and you need to feel each one:
    1. Denser gradient signal per token:
    
        - Each token position now contributes to 3 loss terms instead of 1. The gradient flowing back through position i carries information about tokens i+1, i+2, AND i+3. The model has to develop richer internal representations to satisfy all three objectives simultaneously.

    2. Forces longer-range planning
To predict i+3 from position i, the model must "think ahead" further. This is the same intuition as why chess engines that search deeper play better — more lookahead = better representations.

    3. Better representations for speculative decoding
At inference, you can use the MTP heads to generate draft tokens for speculative decoding. DeepSeek V3 explicitly uses MTP this way — train with MTP, serve with speculative decode. Free inference speedup from the training cost you already paid.

In [3]:
# https://claude.ai/share/1757ca51-af8f-4dd7-9571-6946fd2b4c17

**Full data flow diagram:**
```
input_ids [B, L]           main_hidden [B, L, H]
     │                            │
     │  embed_tokens([:, 1:L-1])  │  curr_hidden = [:, :S', :]
     ▼                            ▼
  [B, S']                    [B, S', H]
     │                            │
  embed_tokens                hidden_norm
  (nn.Embedding)               (RMSNorm)
     │                            │
     ▼                            ▼
  [B, S', H]                 [B, S', H]
     │                            │
  embed_norm                      │
  (RMSNorm)                       │
     │                            │
     └──────────┬─────────────────┘
                │  torch.cat(dim=-1)
                ▼
           [B, S', 2H]
                │
           concat_proj
           (Linear 2H→H)
                │
                ▼
           [B, S', H]
                │
           transformer
           (MLA + FFN)
                │
                ▼
           [B, S', H] ──────────────────► h (passed to next MTP module)
                │
           output_norm
           (RMSNorm)
                │
           lm_head
           (Linear H→V, SHARED)
                │
                ▼
           logits [B, S', V]
                │
      cross_entropy vs target_labels[:, :S']
                │
                ▼
           loss_k (scalar)

In [ ]:
# Step 1: Embed the "hint" tokens (teacher forcing)
t_emb = self.embed_norm(self.embed_tokens(target_tokens))
# embed_tokens: [B=2, S'=4] → lookup → [B=2, S'=4, H=4]
# embed_norm (RMSNorm): [B=2, S'=4, H=4] → [B=2, S'=4, H=4]

# Step 2: Normalize the hidden state from previous depth
h_norm = self.hidden_norm(prev_hidden)
# prev_hidden: [B=2, S'=4, H=4]
# hidden_norm (RMSNorm): [B=2, S'=4, H=4] → [B=2, S'=4, H=4]

# Step 3: Concatenate along feature dim and project
h = self.concat_proj(torch.cat([t_emb, h_norm], dim=-1))
# cat: [B=2, S'=4, H=4] + [B=2, S'=4, H=4] → [B=2, S'=4, 2H=8]
# concat_proj (Linear 2H→H): [B=2, S'=4, 8] → [B=2, S'=4, 4]

# Step 4: Run through transformer block
h = self.transformer(h, attention_mask)
# h: [B=2, S'=4, H=4] → [B=2, S'=4, H=4]

# Step 5: Final norm + lm_head
logits = self.lm_head(self.output_norm(h))
# output_norm: [B=2, S'=4, 4] → [B=2, S'=4, 4]
# lm_head (Linear H→V): [B=2, S'=4, 4] → [B=2, S'=4, V=8]
